In [1]:
import nltk
nltk.download("gutenberg")

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\valla\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


True

In [2]:
from nltk.corpus import gutenberg
import pandas as pd

data = gutenberg.raw('shakespeare-hamlet.txt')

with open('data.txt','w') as f:
    f.write(data)

In [3]:
d = data.lower()
sentences = []
for sent in d.split("\n"):
    sentences.append(sent)


In [4]:
sentences = [sent for sent in sentences if sent!='']

In [5]:
sentences = [sent.strip() for sent in sentences]

In [35]:
sentences[100:200]

["(thereto prick'd on by a most emulate pride)",
 "dar'd to the combate. in which, our valiant hamlet,",
 "(for so this side of our knowne world esteem'd him)",
 "did slay this fortinbras: who by a seal'd compact,",
 'well ratified by law, and heraldrie,',
 'did forfeite (with his life) all those his lands',
 "which he stood seiz'd on, to the conqueror:",
 'against the which, a moity competent',
 "was gaged by our king: which had return'd",
 'to the inheritance of fortinbras,',
 "had he bin vanquisher, as by the same cou'nant",
 'and carriage of the article designe,',
 'his fell to hamlet. now sir, young fortinbras,',
 'of vnimproued mettle, hot and full,',
 'hath in the skirts of norway, heere and there,',
 "shark'd vp a list of landlesse resolutes,",
 'for foode and diet, to some enterprize',
 "that hath a stomacke in't: which is no other",
 '(and it doth well appeare vnto our state)',
 'but to recouer of vs by strong hand',
 'and termes compulsatiue, those foresaid lands',
 'so by h

In [6]:
import re 
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

In [7]:
import pickle
with open("tokenizer","wb") as file:
    pickle.dump(tokenizer,file)

In [8]:
total_words = len(tokenizer.word_index) +1

In [ ]:
tokenizer.word_index

In [10]:
input_sequences = []
for sent in sentences:
    token_list = tokenizer.texts_to_sequences([sent])[0]
    for i in range(1,len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)


In [ ]:
input_sequences

In [12]:
max_seq_len = max([len(sent) for sent in input_sequences])
print(max_seq_len)

14


In [13]:
input_sequences = pad_sequences(input_sequences,max_seq_len,padding='pre')

In [14]:
import numpy as np
input_sequences = np.array(input_sequences)

In [15]:
x,y = input_sequences[:,:-1],input_sequences[:,-1]

In [16]:
x

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       ...,
       [   0,    0,    0, ...,  687,    4,   45],
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4]],
      shape=(25732, 13), dtype=int32)

In [17]:
import tensorflow as tf
y= tf.keras.utils.to_categorical(y,num_classes=total_words)

In [18]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state =42)

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout,Dense,LSTM,GRU,Embedding

model = Sequential()

model.add(Embedding(
    input_dim=total_words,
    output_dim=128
))

model.add(LSTM(64, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(32))
model.add(Dense(total_words, activation='softmax'))

In [20]:
model.compile(optimizer='adam',metrics=['accuracy'],loss='categorical_crossentropy',)

In [21]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(patience=2,monitor='val_loss',)

In [40]:
history = model.fit(
    X_train,
    y_train,
    epochs = 50,
    validation_split=0.2,
    # callbacks = [early_stopping]
)

Epoch 1/50
515/515 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.0641 - loss: 5.7572 - val_accuracy: 0.0454 - val_loss: 7.3975
Epoch 2/50
515/515 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.0678 - loss: 5.6865 - val_accuracy: 0.0486 - val_loss: 7.4447
Epoch 3/50
515/515 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.0748 - loss: 5.6161 - val_accuracy: 0.0493 - val_loss: 7.5537
Epoch 4/50
515/515 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.0797 - loss: 5.5475 - val_accuracy: 0.0522 - val_loss: 7.5848
Epoch 5/50
515/515 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.0882 - loss: 5.4773 - val_accuracy: 0.0522 - val_loss: 7.7005
Epoch 6/50
515/515 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.0909 - loss: 5.4118 - val_accuracy: 0.0522 - val_loss: 7.7637
Epoch 7/50
515/515 ━━━━━━━━━━━━━━━━━━━━ 15s 29ms/step - accuracy: 0.0939 - loss: 5.3497 - val_accuracy: 0.0513 - val_loss: 7.8274
Epoch 8/50
515/515 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.0993 - loss: 5.2863 - v

In [45]:
model.save("lstm.keras")

In [41]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 13, 128)             │         616,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 13, 64)              │          49,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 13, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 4818)                │         158,994 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,512,568 (9.58 MB)

 Trainable params: 837,522 (3.19 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,675,046 (6.39 MB)

In [42]:

import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_next_word(model, tokenizer, text, max_seq_len):
    token_list = tokenizer.texts_to_sequences([text])[0]

    # Handle input containing no known words
    if not token_list:
        return None

    # Keep only the most recent words if the sequence is too long
    token_list = token_list[-(max_seq_len - 1):]

    # Pad the input to the expected length
    token_list = pad_sequences(
        [token_list],
        maxlen=max_seq_len - 1,
        padding='pre'
    )

    # Predict probabilities for the next word
    predicted = model.predict(token_list, verbose=0)

    # Get the index with the highest probability
    predicted_word_index = int(np.argmax(predicted, axis=1)[0])

    # Convert the predicted index back to a word
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word

    return None


input_text = "There is a new car outside"
print(predict_next_word(model, tokenizer, input_text, max_seq_len))


bone


In [36]:

import numpy as np

def debug_prediction(model, tokenizer, text, max_seq_len):
    token_list = tokenizer.texts_to_sequences([text])[0]

    if not token_list:
        print("No known words in the input!")
        return

    token_list = token_list[-(max_seq_len - 1):]

    padded = pad_sequences(
        [token_list],
        maxlen=max_seq_len - 1,
        padding="pre"
    )

    probs = model.predict(padded, verbose=0)[0]

    # Display the 10 most likely next words
    top_indices = np.argsort(probs)[-10:][::-1]

    for idx in top_indices:
        word = tokenizer.index_word.get(idx, "<unknown>")
        print(f"{word:15s} {probs[idx]:.4f}")


and             0.0885
of              0.0881
to              0.0413
in              0.0357
the             0.0197
that            0.0175
is              0.0159
with            0.0156
for             0.0132
a               0.0123


In [44]:
debug_prediction(
    model, tokenizer,
    "for foode and",
    max_seq_len
)

diet            0.0641
fro             0.0328
sweat           0.0313
bulke           0.0256
helpfull        0.0243
repose          0.0214
boy             0.0201
reaches         0.0197
outward         0.0193
filme           0.0148


In [43]:
print("Final training loss:",
      history.history["loss"][-1])

print("Final validation loss:",
      history.history["val_loss"][-1])

print("Final training accuracy:",
      history.history["accuracy"][-1])

print("Final validation accuracy:",
      history.history["val_accuracy"][-1])

Final training loss: 3.5308854579925537
Final validation loss: 10.255090713500977
Final training accuracy: 0.2898348271846771
Final validation accuracy: 0.04906485229730606
